In [11]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline

In [12]:
remove_pitch_types = ['EP', 'FA', 'PO', 'IN', 'UN']
min_pitch_ct = 30
min_years = 3
keep_cols = ['release_speed', 'release_spin_rate', 'release_pos_x', 'release_pos_z', 'release_extension', 'spin_axis']
k_vals = [3,5,7,9,11,15]

In [13]:
def clean_data(filename):
    df = pd.read_csv(filename)
    df['game_date'] = pd.to_datetime(df['game_date'])
    df['year'] = df['game_date'].dt.year.astype(int)
    df = df.sort_values('game_date').reset_index(drop=True)
    keep_cols = ['release_speed', 'release_spin_rate', 'release_pos_x', 'release_pos_z', 'release_extension', 'spin_axis', 'player_name', 'game_date', 'year', 'pitch_type', 'spin_axis']
    df = df.dropna(subset=keep_cols)
    return df


In [16]:
dist_metrics = ['euclidean', 'manhattan']

def prep_data(df, pitcher_name):
    df = df[df['player_name'] == pitcher_name].copy()
    df = df[~df['pitch_type'].isin(remove_pitch_types)]
    count = df['pitch_type'].value_counts()
    valid_types = count[count >= min_pitch_ct].index
    df = df[df['pitch_type'].isin(valid_types)]

    if df.empty or df['pitch_type'].nunique() < 2:
        return None, None, None, None
    
    X = df[keep_cols].values
    y = df['pitch_type'].values
    years = df['year'].values
    return X, y, list(valid_types), years

def forward_chain(years):
    unique_years = sorted(np.unique(years))
    if len(unique_years) < min_years:
        return

    for i in range(1, len(unique_years)):
        train_years = unique_years[:i]
        test_year = unique_years[i]
        train_idx = np.where(np.isin(years, train_years))[0]
        test_idx = np.where(years == test_year)[0]

        if len(train_idx) == 0 or len(test_idx) == 0:
            continue
        yield train_idx, test_idx, train_years, test_year

def select_k(X, y, years, k_vals):
    fold = {k: [] for k in k_vals}
    best_params = {
        'k': k_vals[0],
        'metric': dist_metrics[0]
    }
    best_score = -1.0
    fold = {(k,m): [] for k in k_vals for m in dist_metrics}

    splits = list(forward_chain(years))
    if not splits:
        split = int(len(X)*0.7)
        splits = [(np.arange(split), np.arange(split, len(X)), ['all'], 'holdout')]

    for k in k_vals:
        for metric in dist_metrics:
            fold_scores = []    
            for train_idx, test_idx, train_years, test_year in splits:
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = y[train_idx], y[test_idx]

                if len(np.unique(y_train)) < 2:
                    continue

                pipeline = Pipeline([
                    ('scaler', StandardScaler()),
                    # ('pca', PCA(n_components=0.95)),
                    ('knn', KNeighborsClassifier(n_neighbors=k, metric=metric, weights='distance'))
                ])
                
                pipeline.fit(X_train, y_train)
                score = f1_score(y_test, pipeline.predict(X_test), average='weighted', zero_division=0)

                fold_scores.append(score)
                fold[(k, metric)].append({
                    'train_years': list(train_years),
                    'test_year': test_year,
                    'n_train': len(train_idx),
                    'n_test': len(test_idx),
                    'weighted_f1': round(score, 3)
                })

            if fold_scores:
                mean_score = np.mean(fold_scores)
                if mean_score > best_score:
                    best_score = mean_score
                    best_params = {'k': k, 'metric': metric}
    return best_params, fold

def train_and_eval(X, y, best_params):
    split = int(len(X)*0.7)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    if len(np.unique(y_train)) < 2 or len(X_test)==0:
        return None, None, None
    
    k = best_params['k']
    metric = best_params['metric']

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        # ('pca', PCA(n_components=0.95)),
        ('knn', KNeighborsClassifier(n_neighbors=k, metric=metric, weights='distance'))
    ])

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    weighted_f1 = f1_score(y_test, preds, average='weighted', zero_division=0)
    report = classification_report(y_test, preds, zero_division=0)
    return weighted_f1, report, pipeline

def run_experiment(filepath, output_csv='knn_results.csv', folds_csv='knn_folds.csv'):
    df = clean_data(filepath)
    pitchers = df['player_name'].unique()

    res = []
    all_folds = []

    for pitcher in pitchers:
        X, y, pitch_types, years = prep_data(df, pitcher)
        if X is None:
            print(f'Skipping {pitcher}...not enough data or pitch types')
            continue
        n_years = len(np.unique(years))
        if n_years < min_years:
            print(f'{pitcher} only has data for {n_years} years')

        best_params, fold_info = select_k(X, y, years, k_vals)
        weighted_f1, report, pipeline = train_and_eval(X, y, best_params)

        if weighted_f1 is None:
            print(f'Skipping {pitcher}...not enough data for training/testing')
            continue

        k = best_params['k']
        metric = best_params['metric']

        for fold in fold_info[(k, metric)]:
            all_folds.append({
                'pitcher': pitcher, 
                'test_year': fold['test_year'],
                'weighted_f1': fold['weighted_f1'],
                'n_train': fold['n_train'],
                'n_test': fold['n_test']
            })

            train_str = str(fold['train_years'])
            print(f'train={train_str} | test={fold["test_year"]} | n_train={fold["n_train"]} | n_test={fold["n_test"]} | weighted_f1={fold["weighted_f1"]}')
        
        print(f'\n{'='*55}')
        print(f'Pitcher: {pitcher}')
        print(f'Years of data: {[int(y) for y in sorted(np.unique(years))]}')
        print(f'Pitch types: {pitch_types}')
        print(f'Best k: {k} | Best Metric: {metric}')
        print(f'Weighted F1: {weighted_f1:.3f}')
        print(f"\n{report}")

        print(f'Forward-chain folds (k={k}, metric={metric}):')

        res.append({
            'pitcher': pitcher,
            'pitch_types': ','.join(pitch_types),
            'years': str([int(y) for y in sorted(np.unique(years))]),
            'best_k': k,
            'best_metric': metric,
            'n_years': n_years,
            'n_pitches': len(X),
            'weighted_f1': round(weighted_f1, 3)
        })

        summary_df = pd.DataFrame(res).sort_values('weighted_f1', ascending=False)
        summary_df.to_csv(output_csv, index=False)
        print(f'\nSummary saved to {output_csv}')
    
    pd.DataFrame(all_folds).to_csv(folds_csv, index=False)
    print(f'Fold details saved to {folds_csv}')
    return summary_df

In [17]:
run_experiment('pitcher_data_detailed_cleaned.csv', 'knn_results.csv')

train=[np.int64(2016)] | test=2017 | n_train=2093 | n_test=2851 | weighted_f1=0.944
train=[np.int64(2016), np.int64(2017)] | test=2018 | n_train=4944 | n_test=2713 | weighted_f1=0.947
train=[np.int64(2016), np.int64(2017), np.int64(2018)] | test=2019 | n_train=7657 | n_test=2574 | weighted_f1=0.738
train=[np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)] | test=2020 | n_train=10231 | n_test=1149 | weighted_f1=0.96
train=[np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)] | test=2021 | n_train=11380 | n_test=1727 | weighted_f1=0.97
train=[np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)] | test=2022 | n_train=13107 | n_test=1842 | weighted_f1=0.979
train=[np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)] | test=2023 | n_train=14949 | n_test=1972 | weighted_f1=0.907
train=[np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.

,pitcher,pitch_types,years,best_k,best_metric,n_years,n_pitches,weighted_f1
35,"Miller, Mason","FF,SL,CH,FC","[2023, 2024, 2025]",5,manhattan,3,2512,0.993
8,"deGrom, Jacob","FF,SL,CH,SI,CU","[2016, 2017, 2018, 2019, 2020, 2021, 2022, 202...",11,manhattan,10,17403,0.982
25,"McKenzie, Triston","FF,SL,CU,CH","[2020, 2021, 2022, 2023, 2024, 2025]",9,manhattan,6,6893,0.973
30,"Sánchez, Cristopher","SI,CH,SL","[2021, 2022, 2023, 2024, 2025]",3,euclidean,5,8003,0.965
3,"Verlander, Justin","FF,SL,CU,CH,ST,FC","[2016, 2017, 2018, 2019, 2022, 2023, 2024, 2025]",3,manhattan,8,24506,0.959
9,"Gausman, Kevin","FF,FS,SL,CH,SI","[2016, 2017, 2018, 2019, 2020, 2021, 2022, 202...",5,manhattan,10,26386,0.951
4,"Scherzer, Max","FF,SL,CH,CU,FC","[2016, 2017, 2018, 2019, 2020, 2021, 2022, 202...",5,euclidean,10,23801,0.951
28,"Gilbert, Logan","FF,SL,FS,KC,CU,CH,FC,SI","[2021, 2022, 2023, 2024, 2025]",5,manhattan,5,13106,0.931
31,"Detmers, Reid","FF,SL,CU,CH,SI,ST","[2021, 2022, 2023, 2024, 2025]",15,manhattan,5,7414,0.929
21,"Cease, Dylan","FF,SL,KC,CH,ST,SI","[2019, 2020, 2021, 2022, 2023, 2024, 2025]",5,manhattan,7,17614,0.919


In [26]:
df = clean_data('pitcher_data_detailed_cleaned.csv')
print(df['player_name'].unique())
print(df.shape)


['Kershaw, Clayton' 'Sale, Chris' 'Gibson, Kyle' 'Verlander, Justin'
 'Scherzer, Max' 'Nola, Aaron' 'Walker, Taijuan' 'Cole, Gerrit'
 'deGrom, Jacob' 'Gausman, Kevin' 'Manaea, Sean' 'Darvish, Yu'
 'Anderson, Tyler' 'Lugo, Seth' 'Glasnow, Tyler' 'Wheeler, Zack'
 'Buehler, Walker' 'Mikolas, Miles' 'Almonte, Yency' 'Burnes, Corbin'
 'Suarez, Ranger' 'Cease, Dylan' 'Webb, Logan' 'Singer, Brady'
 'Skubal, Tarik' 'McKenzie, Triston' 'Houck, Tanner' 'Crochet, Garrett'
 'Gilbert, Logan' 'Ober, Bailey' 'Sánchez, Cristopher' 'Detmers, Reid'
 'Sears, JP' 'Kirby, George' 'Pepiot, Ryan' 'Miller, Mason'
 'Miller, Bryce' 'Pfaadt, Brandon' 'Skenes, Paul']
(556513, 21)
